# 12. SOTA Replication: Frozen Multi-Scale Convolutions
**Objective:** Bypass end-to-end CNN training by utilizing frozen random kernels to project VMD Intrinsic Mode Functions into a highly expressive, shift-invariant feature space.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.abspath('../'))
from src.config import PREPROCESSED_DIR
from src.frozen_vmd import extract_frozen_features
from src.classification import train_svm_classifier, evaluate_classifier

plt.style.use('seaborn-v0_8-whitegrid')

### 1. Load VMD Tensor

In [ ]:
vmd_h5_path = os.path.join(PREPROCESSED_DIR, "DB1_subject_1_VMD.h5")
with h5py.File(vmd_h5_path, 'r') as f:
    X_vmd = np.array(f['X_vmd'])
    y_bal = np.array(f['y']).astype(np.int64)
    reps_bal = np.array(f['reps'])

train_reps = [1, 2, 3, 4, 5, 6, 7]
test_reps = [8, 9, 10]

train_idx = np.where(np.isin(reps_bal, train_reps))[0]
test_idx = np.where(np.isin(reps_bal, test_reps))[0]

### 2. Extract Frozen Multi-Scale Features
This projects the (3, 20, 10) physical IMFs into a flat 384-D vector.

In [ ]:
X_frozen_features = extract_frozen_features(X_vmd)

X_train_froz = X_frozen_features[train_idx]
y_train = y_bal[train_idx]

X_test_froz = X_frozen_features[test_idx]
y_test = y_bal[test_idx]

print(f"Frozen Training Features: {X_train_froz.shape}")
print(f"Frozen Testing Features: {X_test_froz.shape}")

### 3. Classify with Support Vector Machine
Because the features are already extracted and statistically pooled, an SVM will optimize the hyperplanes much faster and more reliably than a deep learning backpropagation loop on this dataset size.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_froz)
X_test_scaled = scaler.transform(X_test_froz)

print("\n--- Training SVM on Frozen VMD Features ---")
svm_frozen = train_svm_classifier(X_train_scaled, y_train, kernel="rbf", C=10.0)
results_frozen = evaluate_classifier(svm_frozen, X_test_scaled, y_test)

print(f"Frozen VMD + SVM Accuracy: {results_frozen['accuracy'] * 100:.2f}%")
print(f"Frozen VMD + SVM Macro F1: {results_frozen['macro_f1'] * 100:.2f}%")